# Stack the climate-linked Nigeria DHS households

This notebook appends the 2013, 2018 and 2024 household outputs row by row. The same cluster or household number can occur in different rounds. The combination of `survey_round`, `hv001` and `hv002` must identify one record.

### Before running

In **JupyterLab**, create a folder called `climate_stack_inputs` beside this notebook. Copy the three compressed household files into that folder and choose **Run → Run All Cells**. Keep the `.csv.gz` files compressed.

In **Colab**, upload this notebook and choose **Runtime → Run all**. Select the 2013, 2018 and 2024 household files one at a time when prompted.

| Round | Input file |
|---|---|
| 2013 | `Nigeria_DHS_2013_Households_Climate.csv.gz` |
| 2018 | `Nigeria_DHS_2018_Households_Climate.csv.gz` |
| 2024 | `Nigeria_DHS_2024_Households_Climate.csv.gz` |


## 1. Read and check each round

The code checks round labels, household counts, unique IDs and the three-month exposure windows. The 2024 export predates the status columns in the older-round notebooks. This step uses its GPS flag and climate values to assign the same status categories. The original survey weights are carried through.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

INPUT_DIR = Path("/content/climate_stack_inputs" if files else "climate_stack_inputs")
OUTPUT_DIR = Path("/content/climate_stack_outputs" if files else "climate_stack_outputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Expected counts from the three completed climate QC reports.
EXPECTED = {
    2013: (38522, 307, 307, 575),
    2018: (40427, 207, 207, 322),
    2024: (40047, 0, 0, 232),
}
COUNT_COLS = [
    "households", "households_missing_gps",
    "households_missing_rain_total", "households_missing_temperature_total",
]
KEY = ["survey_round", "hv001", "hv002"]
EXPOSURES = {"rain_mm_3m": "rain_status", "temp_mean_c_3m": "temp_status"}
REQUIRED = {*KEY, "interview_month", "exposure_start", "exposure_end", "gps_usable", "hh_weight", *EXPOSURES}

frames = []
for year, expected in EXPECTED.items():
    filename = f"Nigeria_DHS_{year}_Households_Climate.csv.gz"
    path = INPUT_DIR / filename
    if not path.is_file() and files is not None:
        print(f"Select {filename}", flush=True)
        partial = path.with_suffix(".gz.part")
        try:
            files.upload_file(str(partial))
            if partial.is_file() and partial.stat().st_size:
                partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    if not path.is_file():
        raise FileNotFoundError(f"Place {filename} in {INPUT_DIR.resolve()} and rerun.")

    data = pd.read_csv(path, low_memory=False, compression="gzip")
    if missing := REQUIRED.difference(data.columns):
        raise ValueError(f"{filename} is missing columns: {sorted(missing)}")
    if len(data) != expected[0] or not data.survey_round.eq(year).all():
        raise ValueError(f"{filename} should contain {expected[0]:,} records, all from {year}.")
    if data[KEY].isna().any().any() or data.duplicated(KEY).any():
        raise ValueError(f"{filename} contains missing or duplicate household keys.")

    month = pd.to_datetime(data.interview_month, errors="raise").dt.to_period("M")
    if month.isna().any() or not (
        data.exposure_start.eq((month - 3).astype(str))
        & data.exposure_end.eq((month - 1).astype(str))
    ).all():
        raise ValueError(f"{year}: missing interview month or incorrect climate exposure window.")

    gps = data.gps_usable.astype(str).str.strip().str.lower().map({"true": True, "false": False})
    if gps.isna().any():
        raise ValueError(f"{year}: missing or unrecognised gps_usable flag.")
    data["gps_usable"] = gps
    for value, status in EXPOSURES.items():
        data[value] = pd.to_numeric(data[value], errors="raise")
        if data.loc[~gps, value].notna().any() or not np.isfinite(data[value].dropna()).all():
            raise ValueError(f"{year}: inconsistent GPS or invalid numeric value in {value}.")
        derived = np.select(
            [~gps, data[value].isna()],
            ["missing_gps", "insufficient_coverage"],
            default="available",
        )
        if status in data and not data[status].eq(derived).all():
            raise ValueError(f"{year}: {status} disagrees with the GPS flag or climate value.")
        data[status] = derived
    frames.append(data)
    print(f"Checked {year}: {len(data):,} households")


## 2. Stack, reconcile counts and save

The QC table checks for agreement with the completed round-level climate QC reports. Blank exposures remain blank. The combined file includes all **118,996** households. Keep both output files in your project folder.


In [ ]:
combined = pd.concat(frames, ignore_index=True)
if combined.duplicated(KEY).any():
    raise ValueError("Combined file has duplicate household IDs.")

rows = []
for year, data in combined.groupby("survey_round", sort=True):
    row = {
        "survey_round": year,
        "households": len(data),
        "households_missing_gps": int((~data.gps_usable).sum()),
        "households_low_rain_coverage": int(data.rain_status.eq("insufficient_coverage").sum()),
        "households_low_temperature_coverage": int(data.temp_status.eq("insufficient_coverage").sum()),
        "households_missing_rain_total": int(data.rain_mm_3m.isna().sum()),
        "households_missing_temperature_total": int(data.temp_mean_c_3m.isna().sum()),
    }
    observed = tuple(row[column] for column in COUNT_COLS)
    if observed != EXPECTED[year]:
        raise ValueError(f"{year}: QC does not match the reviewed run. Expected {EXPECTED[year]}, found {observed} ({COUNT_COLS}).")
    rows.append(row)

qc = pd.DataFrame(rows)
qc.loc[len(qc)] = {"survey_round": "TOTAL", **qc.drop(columns="survey_round").sum().to_dict()}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
combined.to_csv(OUTPUT_DIR / "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz", index=False, compression="gzip")
qc.to_csv(OUTPUT_DIR / "Nigeria_DHS_2013_2018_2024_Climate_Stack_QC.csv", index=False)
display(qc)
print(f"Saved both files to {OUTPUT_DIR.resolve()}")
